# MultiLexNorm 2026 — Train ByT5 on Colab

Reproduces the per-language pipeline (MFR baseline → ByT5 → picker) for the W-NUT 2026 MultiLexNorm 2 shared task on a free Colab GPU.

**Before you run:**
1. Runtime → Change runtime type → **GPU** (T4 is free; A100 if you have Pro).
2. Accept dataset access at https://huggingface.co/datasets/weerayut/multilexnorm2026-dev-pub (one click, *Agree and access*).
3. Get a HF read token at https://huggingface.co/settings/tokens — paste it in cell 7 when prompted.

**The fast path (recommended):**
- Run cells 1-12 to set up + run MFR.
- **Skip cells 13-19** (single-language demo + mT5 ablation — optional).
- Run cell 20-21 to launch the main ByT5 training (6 languages, 5 epochs, ~11h without Thai / ~26h with).
- Run cell 22-23 for the final picker comparison.
- Run cell 24-25 to save checkpoints to your Drive.

Repo: https://github.com/Ethan5767/intro-to-ai-assignment

## 1. Verify GPU and clone repo

In [ ]:
!nvidia-smi | head -n 15

In [ ]:
# Repo to clone. Change this if you have your own fork.
REPO_URL = "https://github.com/Ethan5767/intro-to-ai-assignment.git"
REPO_DIR = "intro-to-ai-assignment"

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
%cd $REPO_DIR

## 2. Install dependencies
Colab already ships with `torch` and `transformers`; this just tops up missing pieces and pins compatible versions.

In [ ]:
!pip -q install -r requirements.txt

## 3. Authenticate to Hugging Face
Datasets are gated. Paste your **read** token (starts with `hf_...`) when prompted.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Smoke-check: dataset loads and has the splits we expect
from datasets import load_dataset
ds = load_dataset("weerayut/multilexnorm2026-dev-pub")
print({k: len(v) for k, v in ds.items()})
print("Sample:", ds["train"][0])

## 4. Run unit tests
Should pass in a few seconds; this verifies the data prep, prediction reassembly, and picker logic survive any environment differences.

In [ ]:
!python -m pytest tests/ -q

## 5. MFR baseline — all 17 languages
Fast (CPU-only), no GPU needed. Writes per-language predictions and `outputs/summary_mfr.csv` (consumed by the picker).

In [ ]:
!python run_mfr.py

## 6. (Optional) Quick single-language demo

Trains ByT5 on **one** language with `--limit-train 5000 --epochs 3` so you can verify the pipeline end-to-end in ~15 min before launching the multi-hour main run. **Skip this section if you're going straight to the main run (cells 9-10).**

In [ ]:
LANG = "en"            # one of: en de nl es it hr sr sl da tr id iden trde ja ko th vi
EPOCHS = 3
LIMIT_TRAIN = 5000     # set to -1 for the full split (slower)

!python train_seq2seq.py --model byt5-small --lang $LANG --epochs $EPOCHS --limit-train $LIMIT_TRAIN

## 7. Predict + score on dev


In [ ]:
import os
os.makedirs(f"outputs/byt5/{LANG}", exist_ok=True)

!python predict_seq2seq.py --ckpt ckpts/byt5-small/$LANG/best --lang $LANG --split validation --out-json outputs/byt5/$LANG/predictions_dev.json --batch-size 16 --num-beams 1

In [ ]:
import json
from datasets import load_dataset
from utils import evaluate

ds = load_dataset("weerayut/multilexnorm2026-dev-pub")
items = [dict(x) for x in ds["validation"] if x["lang"] == LANG]
raw = [it["raw"] for it in items]
gold = [it["norm"] for it in items]
with open(f"outputs/byt5/{LANG}/predictions_dev.json", encoding="utf-8") as f:
    preds = [r["pred"] for r in json.load(f)]
evaluate(raw, gold, preds)

## 8. (Optional) Train mT5-small on the same language for the ablation

In [ ]:
!python train_seq2seq.py --model mt5-small --lang $LANG --epochs $EPOCHS --limit-train $LIMIT_TRAIN

## 9. RECOMMENDED PATH — Train ByT5 on the 6 target languages (5 epochs, full data)

This is the main training cell for the assignment. It trains in **ascending size order** so quick wins land first; if Colab disconnects, you still have the smaller languages done.

Estimated runtime on a **free T4 (16 GB)**:

| order | lang | examples | wall-clock |
|---|---|---|---|
| 1 | es | 7 k | ~1 h |
| 2 | nl | 12 k | ~1.7 h |
| 3 | it | 13 k | ~1.7 h |
| 4 | de | 15 k | ~2 h |
| 5 | en | 35 k | ~5 h |
| 6 | th | 140 k | **~15 h** |
| | total | | **~26 h** |

**Free Colab session cap is 12 hours.** Drop `th` from `LANGS` to fit in one session — ByT5 typically *loses* to MFR on Thai anyway because byte-level tokenization fragments multi-byte UTF-8. If you keep `th`, plan to save checkpoints to Drive (cell 12) and resume in a fresh session.

`run_pipeline.py` is idempotent: if a checkpoint already exists for a language, it skips re-training. Safe to re-run after disconnects.

In [ ]:
# Edit LANGS to skip Thai if you're on free Colab (12h session cap):
#   LANGS = "es nl it de en"        # ~11h, fits in one free session
#   LANGS = "es nl it de en th"     # ~26h, needs Pro or Drive-resume
LANGS = "es nl it de en th"
EPOCHS = 5

!python run_pipeline.py --model byt5-small --langs $LANGS --epochs $EPOCHS

## 10. Final results — picker comparison and scoring summary

After cell 21 finishes, the picker compares MFR vs ByT5 dev-ERR per language and chooses the winner with a 2-point min-gap (favouring ByT5). Run this cell to see the verdict.

In [ ]:
# Show per-language ByT5 dev ERR
import pandas as pd, os
for f in ["outputs/summary_mfr.csv", "outputs/summary_byt5-small.csv"]:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"\n=== {f} ===")
        df["err_pct"] = (df["err"] * 100).round(2)
        print(df[["lang", "lai", "acc", "err_pct"]].to_string(index=False))

# Run the picker (compares MFR vs ByT5 vs mT5 if present)
!python picker.py

## 11. Save artefacts to Drive (optional)
Colab wipes its disk between sessions — copy the checkpoint and predictions out if you want to resume later, or to share with teammates.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/multilexnorm2026
!cp -r ckpts /content/drive/MyDrive/multilexnorm2026/ 2>/dev/null || true
!cp -r outputs /content/drive/MyDrive/multilexnorm2026/